# Data Loaders

In [13]:
import os
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader

class PotholeDataset(Dataset):
    def __init__(self, image_dir, label_dir):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.image_files = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_name = self.image_files[idx]
        img_path = os.path.join(self.image_dir, image_name)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # normalize
        image = image.astype(np.float32) / 255.0
        print(image.shape)

        # create empty mask
        mask = np.zeros(image.shape[:-1], dtype=np.float32)

        label_path = os.path.join(self.label_dir, image_name.replace('.jpg', '.txt'))
        with open(label_path, 'r') as f:
            lines = f.readlines()
            for l in lines:
                parts = l.strip().split()
                class_id = int(parts[0])
                coords = np.array([float(x) for x in parts[1:]]).reshape(-1, 2)

                coords[:, 0] *= image.shape[0]
                coords[:, 1] *= image.shape[0]
                coords = coords.astype(np.int32)
                cv2.fillPoly(mask, [coords], color=1.0)
        mask = np.expand_dims(mask, axis=-1)           
        return image, mask

        

In [7]:
def getPotholeDataloader(batch_size, image_dir, label_dir):
    dataset = PotholeDataset(image_dir, label_dir)     
    return DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        drop_last=True, 
        num_workers=4
    )

In [14]:
# Test
dataset = PotholeDataset("/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/images", "/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/labels")     
loader = getPotholeDataloader(4, 
                     "/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/images", 
                     "/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/labels")
dataset[0]

(640, 640, 3)


(array([[[0.16078432, 0.1764706 , 0.22352941],
         [0.16078432, 0.1764706 , 0.22352941],
         [0.16862746, 0.18431373, 0.23137255],
         ...,
         [0.5803922 , 0.5176471 , 0.46666667],
         [0.58431375, 0.52156866, 0.47058824],
         [0.5882353 , 0.5254902 , 0.4745098 ]],
 
        [[0.16470589, 0.18039216, 0.22745098],
         [0.16470589, 0.18039216, 0.22745098],
         [0.16470589, 0.18039216, 0.22745098],
         ...,
         [0.5803922 , 0.5176471 , 0.46666667],
         [0.58431375, 0.52156866, 0.47058824],
         [0.58431375, 0.52156866, 0.47058824]],
 
        [[0.17254902, 0.18431373, 0.24313726],
         [0.16862746, 0.18039216, 0.23921569],
         [0.16470589, 0.1764706 , 0.23529412],
         ...,
         [0.5764706 , 0.50980395, 0.47058824],
         [0.5803922 , 0.5137255 , 0.4745098 ],
         [0.5803922 , 0.5137255 , 0.4745098 ]],
 
        ...,
 
        [[0.36078432, 0.36862746, 0.3647059 ],
         [0.49019608, 0.49411765, 0.50980

# Rotary Position Embeddings (RoPE)

In [ ]:
import jax
import jax.numpy as jnp 
import flax.linen as nn

def apply_rope(x, head_dim):
    # X -> B,S,H,head_dim
    seq_len = x.shape[1]
    dims = jnp.arange(seq_len, dtype=jnp.float32).reshape(-1, 1) # seq_len, 1
    i_2 = jnp.arange(0, head_dim, 2, dtype=jnp.float32)
    div_term = jnp.exp(-1* i_2 * jnp.log(100000)/head_dim) # head_dim//2
    freq = dims * div_term # seq_len, head_dim//2
    sin_freq = jnp.repeat(jnp.sin(freq), 2, axis=-1)
    cos_freq = jnp.repeat(jnp.cos(freq), 2, axis=-1)
    odd_terms = x[..., 1::2]
    even_terms = x[..., 0::2]
    x_tmp = jnp.stack([-odd_terms, even_terms], axis=-1)
    return x * cos_frep + x_tmp * sin_freq
